# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.  We will utilize the parser generator
`Larkita` to define our language rules.  The parser generated by `Larkita` returns the abstract syntax tree of a
program as a nested tuple, so we do not have to convert a parse tree.

In [ ]:
from larkita import Larkita

## Specification of the Grammar

Below is the grammar for our simple `C`-like language expressed as an *Extended Backus-Naur Form* (EBNF) string.

Our programming language allows both single-line (`//`) and multi-line (`/* */`) comments.  We instruct the parser
to entirely `%ignore` them alongside standard whitespace.

Note that we have to use a *raw string* here, because otherwise the escape sequence `\/` would not be allowed.

The grammar also determines the abstract syntax tree:  The string literals of an alternative form the *head* of the
tuple that is returned for this alternative, while all other symbols become its *children*.

* The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
  semicolon from the heads of *all* alternatives, since these literals carry no information.  Hence the alternative
  `"while" "(" bool_expr ")" stmnt` yields the tuple `('while', test, stmnt)` and the alternative
  `IDENTIFIER ":=" expr ";"` yields `(':=', ('IDENTIFIER', name), expr)`.
* An *inner alternative* like `("+" | "-")` abbreviates several alternatives:  The alternative
  `expr ("+" | "-") product` stands for the two alternatives `expr "+" product` and `expr "-" product`.  Hence the
  operator that has been read is the head, e.g. `x - 1` yields `('-', ('IDENTIFIER', 'x'), ('NUMBER', '1'))`.
* An alternative that consists of a single symbol passes the abstract syntax tree of this symbol through.  Hence a
  block `"{" program "}"` simply *is* its program, and a parenthesized expression is the expression inside.
* An *alias* `-> name` replaces the head.  Without the alias, the alternative `expr ";"` would consist of a single
  symbol and would therefore return the expression itself.  With the alias, it yields `('expr', expr)`.
* The arguments of a function call are written inline as `[expr ("," expr)*]`.  The children of an optional part and
  of a repetition are spliced into the tuple of the alternative, so a function call yields
  `('call', ('IDENTIFIER', name), arg_1, ..., arg_n)`.
* The program is the first rule of the grammar and therefore its start symbol.  As the alternative `stmnt*` contains
  no literal, the name of the rule is used as head:  A program yields the tuple `('program', stmnt_1, ..., stmnt_n)`.

In [ ]:
sl_grammar = r"""
program: stmnt*

stmnt: "if" "(" bool_expr ")" stmnt
     | "while" "(" bool_expr ")" stmnt
     | "{" program "}"
     | IDENTIFIER ":=" expr ";"
     | expr ";"                           -> "expr"

bool_expr: expr ("==" | "!=" | "<=" | ">=" | "<" | ">") expr

expr: expr ("+" | "-") product
    | product

product: product ("*" | "/" | "%") factor
       | factor

factor: "(" expr ")"
      | NUMBER
      | IDENTIFIER
      | IDENTIFIER "(" [expr ("," expr)*] ")" -> "call"

// These literals are left out of the heads of all tuples.
%ignore: "(", ")", "{", "}", ";"

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.INT   -> NUMBER
%import common.WS

// In order to show how it's done, I have defined the tokens
// CPP_COMMENT and C_COMMENT describing comments myself.
CPP_COMMENT: /\/\/[^\n]*/
C_COMMENT:   "/*" /(.|\n)*?/ "*/"

// Ignore whitespace and comments
%ignore WS
%ignore C_COMMENT
%ignore CPP_COMMENT
"""

We generate the parser.  `Larkita` always generates a *canonical LR(1) parser*.  A description of the states of the
LR(1) automaton is written into the file `sl.out`.

In [ ]:
parser = Larkita(sl_grammar, out_file='sl.out')

## The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple.  This tuple has exactly the structure that is needed
by our evaluation and visualization functions.  The shapes that are produced are the following:

| construct              | nested tuple                                       |
|:-----------------------|:---------------------------------------------------|
| program / block        | `('program', stmnt, ...)`                          |
| assignment             | `(':=', ('IDENTIFIER', name), expr)`               |
| expression statement   | `('expr', expr)`                                   |
| conditional, loop      | `('if', test, stmnt)`, `('while', test, stmnt)`    |
| binary operator        | `(op, lhs, rhs)`, for example `('+', lhs, rhs)`    |
| number, variable       | `('NUMBER', digits)`, `('IDENTIFIER', name)`       |
| function call          | `('call', ('IDENTIFIER', name), arg, ...)`         |

For example, the statement `x := 1 + 2 * y;` yields the following tuple:

In [ ]:
parser.parse('x := 1 + 2 * y;')

The external helper file `AST2Dot.ipynb` is used to represent our generated nested tuple ADT graphically.

In [ ]:
%run AST2Dot.ipynb

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The type `NestedTuple` describes the nested tuples that represent programs, while `Number` is the type of
the values that are stored in the variables of our programs.

In [ ]:
Number      = int | float
NestedTuple = AST

The function `parse` takes a `file_name` as its sole argument.  The file is read, parsed into a nested tuple, and
visualized via `graphviz`.

In [ ]:
def parse(file_name):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    ast = parser.parse(program)
    print(ast)
    return tuple2dot(ast)

In [ ]:
parse('sum.sl')

The function `execute_tuple` loops through statements linearly. The dictionary `Values` is the continuous state that gets updated as assignments occur.

In [ ]:
def execute_tuple(StatementList, Values={}):
    for stmnt in StatementList:
        execute(stmnt, Values)

The function `execute` matches recursively against the shape of the nested tuple statement node, triggering evaluation or recursive execution sequences.

In [ ]:
def execute(stmnt, Values):
    match stmnt:
        case ('program', *SL):
            execute_tuple(tuple(SL), Values)
        case (':=', ('IDENTIFIER', var), value):
            Values[var] = evaluate(value, Values)
        case ('expr', expr):
            evaluate(expr, Values)
        case ('if', test, stmnt):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('while', test, stmnt):
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
        case _:
            assert False, f'{stmnt} unexpected'

The function `evaluate_bool` executes conditional logical statements returning a pure boolean output.

In [ ]:
def evaluate_bool(expr: NestedTuple, Values: dict[str, Number]) -> bool:
    match expr:
        case ('==', lhs, rhs):
            return evaluate(lhs, Values) == evaluate(rhs, Values)
        case ('!=', lhs, rhs):
            return evaluate(lhs, Values) != evaluate(rhs, Values)
        case ('<=', lhs, rhs):
            return evaluate(lhs, Values) <= evaluate(rhs, Values)
        case ('>=', lhs, rhs):
            return evaluate(lhs, Values) >= evaluate(rhs, Values)
        case ('<', lhs, rhs):
            return evaluate(lhs, Values) <  evaluate(rhs, Values)
        case ('>', lhs, rhs):
            return evaluate(lhs, Values) >  evaluate(rhs, Values)
        case _:
            assert False, f'{expr} unexpected'

The function `evaluate` calculates specific numerical values given either primitives (numbers and variables, which
are represented as tokens) or compound arithmetic operations.  We utilize python's built-in `match/case` to guarantee
exhaustive mapping of our AST types.

In [ ]:
def evaluate(expr: NestedTuple, Values: dict[str, Number]) -> Number:
    match expr:
        case ('NUMBER', digits):
            return int(digits)
        case ('IDENTIFIER', name):
            return Values[name]
        case ('call', ('IDENTIFIER', 'read')):
            return int(input('Please enter a natural number: '))
        case ('call', ('IDENTIFIER', 'print'), expr):
            print(evaluate(expr, Values))
            return 0
        case ('+', lhs, rhs):
            return evaluate(lhs, Values) + evaluate(rhs, Values)
        case ('-', lhs, rhs):
            return evaluate(lhs, Values) - evaluate(rhs, Values)
        case ('*', lhs, rhs):
            return evaluate(lhs, Values) * evaluate(rhs, Values)
        case ('/', lhs, rhs):
            return evaluate(lhs, Values) / evaluate(rhs, Values)
        case ('%', lhs, rhs):
            return evaluate(lhs, Values) % evaluate(rhs, Values)
        case _:
            assert False, f'{expr} unexpected'

In [ ]:
!cat sum.sl

Finally, we bring the whole pipeline together.  `main` will open the program file, parse it into a nested tuple, and
evaluate it under an isolated state dict.

In [ ]:
def main(file):
    with open(file, 'r') as handle:
        program = handle.read()
    ast = parser.parse(program)
    display(tuple2dot(ast))
    Values = {}
    execute(ast, Values)

In [ ]:
main('sum.sl')

In [ ]:
!cat factorial.sl

In [ ]:
main('factorial.sl')